# On The Idea Of *Market Expectations*

A common practice among interest-rate practitioners is to use yield curve, discount factors and forward rate data — for example from bootstrapped OIS curves — as the market’s expectations of future policy rates. For instance, in the case of Chile’s Central Bank, this type of information is often illustrated in their monetary policy reports.

```{image} ./images/policyrate.png
:alt: Corredor TPM: Chile Banco Central
:class: bg-primary
:width: 500px
:align: center
```

A pesar de que algunos podria pensar que existe una profesia autocumplida en esta idea, la realidad es que no existen argumentos para pensar que sea realmente cierta. Matematicamente hablando, hay una diferencia entre lo que son las probabilidades reales de que ocurra un evento de politica monetaria y las probabilidades riesgo neutrales implicitas en los precios. En articulo ahondaré un poco más en que sucede en la practica y cuales son los desafios presentes al tratar de entender que *piensa* el mercado.

## Un modelo simple de tasas de interes

### Model dependency

The discount factors and all other metrics are derived from a **risk-neutral** expectation. The random variable $\exp\{-\int_t^T r_s ds\}$ depends on the short rate process $ r_t $, which is not directly observable in the market. In fact, different dynamics for $r_t$ can lead to the same $P(t,T)$ (there are infinitely many short-rate models that can fit a given yield curve).

To illustrate this idea, let's build a numerical example:

- Data Generation:
  We start by creating a synthetic yield curve from the expected discount factors that arrise from multiple simulations of a short rate process. We want to build a dynamic that behaves as closely as posible to the evolution of the policy rate. For this, we define a pure-jump process as

  $$
      \int_t^T r_s\, ds 
      = r_t (T - t) + \sum_{\tau_i \in (t, T]} J_i (T - \tau_i),
  $$
  where $\bar{r}_n$ denotes the prevailing short rate during the period $[\tau_n,\tau_{n+1})$ and $J_n$ represents the stochastic jump in the policy rate at meeting $\tau_n$. The only source of randomness in this setting is the *policy decision itself*, not continuous micro-fluctuations of the short rate. 

- Model Setup:
  For the model, we will use a Hull-White one-factor model to represent the short-rate dynamics:
    $$
    dr_t = \alpha(\theta(t) - r_t) dt + \sigma dW_t^{\mathbb Q},
    $$
  where $\alpha$ is the speed of mean reversion, $\sigma$ is the volatility, and $\theta(t)$ is a time-dependent mean reversion level that we will calibrate to fit the observed yield curve.This functions is given by
    $$
    \theta(t) = \frac{\partial f(0,t)}{\partial t} + \alpha f(0,t) + \frac{\sigma^2}{2\alpha}(1 - e^{-2\alpha t}).
    $$  
  The expected value of $r_t$ under $\mathbb Q$ is
    $$
    \mathbb{E}^{\mathbb Q}[r_t] = r_0 e^{-\alpha t} + \int_0^t \alpha \theta(s) e^{-\alpha(t-s)} ds.
    $$
  Of course, different specifications of $r_t$ will produce different $\mathbb{E}^{\mathbb Q}[r_t]$. Notice that in this particular case the expectation is dependent on the parameters $a$ and $\sigma$, which clearly are not parameters required OIS swaps today.

In [ ]:
from scipy.interpolate import UnivariateSpline
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import plotly
from IPython.display import display, HTML
from scipy.optimize import minimize, LinearConstraint
import pandas as pd 
from dataclasses import dataclass, field


### Simulation

We begin by simulating a policy rate process. We assume that there are quarterly meetings where the rate can jump up or down by a fixed amount, otherwise it remains constant. This is a simple way to mimic the behavior of central banks that adjust rates based on economic conditions.

In [ ]:
# Simulation parameters
YEARS = 10
STEPS_PER_YEAR = 360
DT = 1.0 / STEPS_PER_YEAR
R0 = np.array([0.03])
MEETING_DATES = np.arange(0.25, YEARS + 0.25, 0.25)  

# Jump size
J = 0.0025 
INTENSITIES = np.arange(-4*J,5*J, J)

# utility to generate probs that sum up to 1
def gen_probs(n_states: int):
    u = np.random.uniform(0, 1,size=n_states)
    return u/u.sum()

@dataclass
class Curve:
    times: np.array
    discounts: np.array

    def discount(self, t):
        return np.interp(t, self.times, self.discounts)
    
    def fwd_rate(self, t, T):
        df1 = self.discount(t)
        df2 = self.discount(T)
        return -np.log(df2/df1)/(T-t)
    
@dataclass
class Simulation:
    times: np.ndarray
    rate_paths: np.ndarray
    discount_paths: np.ndarray
    expected_rates: np.ndarray
    expected_discounts: np.ndarray
    probs: np.ndarray

    def get_curve(self):
        return Curve(self.times, self.expected_discounts)
    
@dataclass
class SimpleJumpSimulation:
    intensities: np.ndarray
    meeting_dates: np.ndarray
    dt: float
    r0: float
    same_prob: bool = True  

    def gen_paths(self, n_paths=10_000):
        horizon = float(self.meeting_dates[-1])
        steps = int(round(horizon / self.dt))
        times = np.arange(steps + 1) * self.dt  

        meeting_idx = np.round(self.meeting_dates / self.dt).astype(int)
        M = meeting_idx.size
        K = self.intensities.size

        increments = np.zeros((n_paths, steps + 1))
        probs = np.zeros((M, K))

        if self.same_prob:
            p_all = gen_probs(K)
            probs[:] = p_all
            for i, idx in enumerate(meeting_idx):
                increments[:, idx] = np.random.choice(self.intensities, size=n_paths, p=p_all)
        else:
            for i, idx in enumerate(meeting_idx):
                p_i = gen_probs(K)
                probs[i] = p_i
                increments[:, idx] = np.random.choice(self.intensities, size=n_paths, p=p_i)

        rate_paths = self.r0 + np.cumsum(increments, axis=1)

        acc_int_paths = np.zeros_like(rate_paths)
        acc_int_paths[:, 1:] = np.cumsum(rate_paths[:, :-1] * self.dt, axis=1)

        discount_paths = np.exp(-acc_int_paths)
        expected_rates = rate_paths.mean(axis=0)
        expected_discounts = discount_paths.mean(axis=0)

        return Simulation(times, rate_paths, discount_paths, expected_rates, expected_discounts, probs)

In [ ]:
# Simulation
engine = SimpleJumpSimulation(INTENSITIES, MEETING_DATES, DT, R0)
n_paths = 1_000
simulation = engine.gen_paths()

show_k = 60  
idx_subset = np.random.choice(n_paths, size=min(show_k, n_paths), replace=False)

# Plotting
fig = make_subplots(rows=1, cols=2, subplot_titles=("Short-Rate Simulations", "Expected Discounts"))
for k in idx_subset:
    fig.add_trace(go.Scatter(x=simulation.times, y=simulation.rate_paths[k], mode='lines', line=dict(width=1, color='blue'), opacity=0.1, showlegend=False, hoverinfo='skip'), row=1, col=1)
fig.add_trace(go.Scatter(x=simulation.times, y=simulation.expected_rates, mode='lines', name='Expected Short Rate', line=dict(width=1, color='red')), row=1, col=1)
fig.add_trace(go.Scatter(x=simulation.times, y=simulation.expected_discounts, mode='lines',name='Expected Discounts', line=dict(width=1)),row=1, col=2)
fig.update_yaxes(tickformat=".2%", row=1, col=1, title_text="Rate")
fig.update_yaxes(row=1, col=2, title_text="Discount factor")
fig.update_xaxes(title_text="Time (years)", row=1, col=1)
fig.update_xaxes(title_text="Time (years)", row=1, col=2)
fig.update_layout(title="Short-Rate Simulations and Expected Discounts", showlegend=True, legend=dict(orientation="h", yanchor="top", y=-0.1, xanchor="center", x=0.5))
fig.show()

pd.DataFrame(simulation.probs, index=np.round(MEETING_DATES, 6), columns=[f"{s*1e4:+.0f} bp" for s in INTENSITIES])

In [ ]:
eps = 1e-12
lnP_obs = np.log(simulation.expected_discounts)
f_raw = np.zeros_like(simulation.times)
f_raw[1:] = -(lnP_obs[1:] - lnP_obs[:-1]) / DT

s_factor = 5e-4
mask_pos = simulation.times > 0
spl_f = UnivariateSpline(simulation.times[mask_pos], f_raw[mask_pos], s=s_factor)
f = np.zeros_like(simulation.times)
f[mask_pos] = spl_f(simulation.times[mask_pos])
fprime = np.zeros_like(simulation.times)
fprime[mask_pos] = spl_f.derivative()(simulation.times[mask_pos])

r0 = f[1]

def hw_theta_from_curve(t, f, fprime, a, sigma):
    theta = np.zeros_like(t)
    theta[0] = f[1]  # benign at 0
    theta[1:] = fprime[1:] + a * f[1:] + (sigma**2)/(2*a) * (1.0 - np.exp(-2.0 * a * t[1:]))
    return theta

def hw_expectation_Q(t, theta, a, r0):
    E = np.zeros_like(t)
    E[0] = r0
    I = 0.0
    decay = np.exp(-a * DT)
    for k in range(1, len(t)):
        I = decay * I + a * 0.5 * DT * (theta[k] + decay * theta[k-1])
        E[k] = r0 * np.exp(-a * t[k]) + I
    return E

param_sets = [
    (0.20, 0.005),
    (0.50, 0.010),
    (1.00, 0.015),
]

cum_int_f = np.zeros_like(simulation.times)
cum_int_f[1:] = np.cumsum(0.5 * (f[:-1] + f[1:]) * DT)
P_fit = np.exp(-cum_int_f)
y_fit = np.zeros_like(simulation.times); y_fit[1:] = -np.log(P_fit[1:]) / simulation.times[1:]; y_fit[0] = r0

mean_paths = []
for a_hw, sigma_hw in param_sets:
    theta = hw_theta_from_curve(simulation.times, f, fprime, a_hw, sigma_hw)
    E_hw = hw_expectation_Q(simulation.times, theta, a_hw, r0)
    mean_paths.append((a_hw, sigma_hw, E_hw))


# to plot only some points
mask_t = (simulation.times * STEPS_PER_YEAR) % 180 == 0

# Plotting
fig = make_subplots(rows=1, cols=2, subplot_titles=("Discounts (observed vs fitted)", "E[r_t]"))
for a_hw, sigma_hw, _ in mean_paths:
    fig.add_trace(go.Scatter(x=simulation.times[mask_t], y=P_fit[mask_t], mode="markers+lines", name=f"Fitted-α={a_hw:.1f}, σ={sigma_hw:.2%}", line=dict(width=2, dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=simulation.times[mask_t], y=simulation.expected_discounts[mask_t], name="Observed discounts", mode="lines", line=dict(color="green", width=1)), row=1, col=1)

for a_hw, sigma_hw, E_hw in mean_paths:
    fig.add_trace(go.Scatter(x=simulation.times, y=E_hw, mode="lines", name=f"E[r_t] — α={a_hw:.1f}, σ={sigma_hw:.2%}"), row=1, col=2)

for c in (1,2):
    fig.update_yaxes(tickformat=".2%", row=1, col=c)
    fig.update_xaxes(title_text="Time (years)", row=1, col=c)

fig.update_yaxes(title_text="Discount factor", row=1, col=1)
fig.update_yaxes(title_text="Short rate (mean)", row=1, col=2)
fig.update_layout(title="Hull-White Model", legend_title="Series")
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.3, xanchor="center", x=0.5))
fig.show()

As expected, the Hull-White model is able to fit perfectly the yield curve at time $t=0$, but $\mathbb{E}^{\mathbb Q}[r_t]$ varies significantly for different values of $\alpha$ and $\sigma$.

# Overcoming this limitations

## A jump-only model for the short-rate process

We begin by considering the behavior of the short rate at the present time. 
Policy rates are known today and are adjusted by central banks only at scheduled monetary policy meetings. 
As a result, the short rate remains **constant** between meetings. 
When a new policy decision is announced, the rate changes in discrete steps, typically in multiples of a fixed increment $\delta$. 
We model this as

$$
    r_t = \bar{r}_n, \qquad 
    \bar{r}_n = \bar{r}_{n-1} + J_n, \qquad 
    \tau_n \le t < \tau_{n+1}, \qquad 
    \bar{r}_0 = r_0,
$$

where $\bar{r}_n$ denotes the prevailing short rate during the period $[\tau_n,\tau_{n+1})$ and $\{J_i\} = \{\dots, J^{\text{Up}}, J^{\text{Stay}}, J^{\text{Down}}, \dots\}$  represents the stochastic jump in the policy rate at meeting $\tau_n$. The only source of randomness in this setting is the *policy decision itself*, not continuous micro-fluctuations of the short rate. 
Consequently, no diffusion or volatility component is required. 
We also assume there are no unscheduled *surprise* meetings, so the sequence of meeting dates $\{\tau_n\}$ is known in advance. Under this specification, the integral of the short rate between $t$ and $T$ is

$$
    \int_t^T r_s\, ds 
    = r_t (T - t) + \sum_{\tau_i \in (t, T]} J_i (T - \tau_i),
$$

and lets define $\Delta \tau_i := T - \tau_i$ the length of time during which the new policy rate applies after the jump at $\tau_i$ and before maturity $T$. Then, the discount factor can be expressed as

$$
    P(t,T) 
        = e^{-r_t (T - t)} \cdot \mathbb{E}^{\mathbb{Q}}\left[e^{-\sum_{\tau_i \in (t, T]} J_i \Delta \tau_i} \Big| \mathcal{F}_t \right].
$$

Now, we have two options to proceed. If we assume that the jumps $\{J_i\}$ are independent and identically distributed (i.i.d.) random variables, we can get the expectation inside the product, making the calculation much easier. 

Check claims:
The problem with this approach is that we are assuming that the jumps do not depend on the last known rate, which is not realistic; we know that if a big hike just happened, the next jump is more likely to be smaller. If we assume that the jumps $\{J_i\}$ are not independent, we need to come up with a recursive way to compute the expectation. Let's see both cases.

### Independent Jumps

Assuming the jumps $\{J_i\}$ are conditionally independent given $\mathcal{F}_t$, the discount factor can be written as

$$
    P(t,T) 
    = \mathbb{E}^{\mathbb{Q}}\left[e^{-\int_t^T r_s\, ds} \Big| \mathcal{F}_t\right]
    = e^{-r_t (T - t)} \prod_{\tau_i \in (t, T]} \mathbb{E}^{\mathbb{Q}}\left[e^{-J_i \Delta \tau_¡} \Big| \mathcal{F}_t\right].
$$

Using the definition of instantaneous forward rate, for maturities $T$ that do not coincide with meeting dates, differentiation gives

$$
    f(t,T)
    = r_t-\sum_{\tau_i \in (t, T]} 
    \frac{\partial}{\partial T} \ln 
    \mathbb{E}^{\mathbb{Q}}\left[e^{-J_i \Delta \tau_¡} \Big| \mathcal{F}_t\right] = r_t + \sum_{\tau_i \in (t, T]} 
    \frac{\mathbb{E}^{\mathbb{Q}}\left[J_i\, e^{-J_i\Delta \tau_¡} \,\big|\, \mathcal{F}_t\right]}
    {\mathbb{E}^{\mathbb{Q}}\left[e^{-J_i\Delta \tau_¡ } \,\big|\, \mathcal{F}_t\right]}.
$$

As the dynamics of the short rate depend on our modeling choices, a possible approach that avoids this could be to learn from data the dynamics of the process, for example using a neural SDE model or other ML techniques. The problem of this is that, to my knowledge, there aren't yet methods that accodate jump dynamics in a sensible way. 

My take here is to model the evolution of the policy rate as close as possible to the actual behaviour of it, which surprisingly is not that complex in many geografies. Below is a sumary of the ideas behind a simplified model.

In [ ]:
@dataclass
class SimpleJumpsModel:
    r0: np.ndarray
    meeting_dates: np.ndarray
    intensities: np.ndarray
    probs: np.ndarray = field(default=np.array)

    def __expectation(self, t):
        meetings_before_t = self.meeting_dates[self.meeting_dates < t]
        yf_meetings = t - meetings_before_t
        s = 0.0
        for i, yf in enumerate(yf_meetings):
            p = self.probs[i, :]
            w = np.exp(-yf * self.intensities)                   
            m0 = np.dot(p, w)                                    
            m1 = np.dot(p, self.intensities * w)                            
            s += m1 / m0
        return s

    def inst_fwd_rate(self, t):
        return self.r0 + self.__expectation(t)

    def discount(self, t):
        s = -self.r0 * t
        for i, tau_i in enumerate(self.meeting_dates):
            if tau_i < t - 1e-12:
                u = t - tau_i
                p = self.probs[i, :]
                m0 = np.dot(p, np.exp(-u * self.intensities))
                m0 = max(m0, 1e-18)
                s += np.log(m0)
        return np.exp(s)

    def fwd_rate(self, t, T):
        df1 = self.discount(t)
        df2 = self.discount(T)
        return -np.log(df2 / df1) / (T - t)

    def fit(self, curve: Curve, shift=0.1):
        knots = self.meeting_dates + shift
        M, K = self.meeting_dates.shape[0], self.intensities.shape[0]
        self.probs = np.zeros((M, K))

        cons = [{'type': 'eq', 'fun': lambda p: np.sum(p) - 1.0}]
        bnds = [(0.0, 1.0)] * K
        for i, knot in enumerate(knots):
            z0 = gen_probs(self.intensities.shape[0])
            def obj(p):                
                old = self.probs[i, :].copy()
                self.probs[i, :] = p
                err = curve.discount(knot) - self.discount(knot)
                self.probs[i, :] = old
                return err**2

            res = minimize(obj, z0, bounds=bnds, constraints=cons, options=dict(maxiter=10000, ftol=1e-15))
            self.probs[i, :] = res.x

    def get_probs(self):
        columns=[f"{s*1e4:+.0f} bp" for s in self.intensities]
        df = pd.DataFrame(self.probs, columns=columns, index=self.meeting_dates).round(2)
        return df
    
    def __repr__(self):
        return self.get_probs().to_string()
    
max_t = 2.2 # slightly higher t so we fit correctly up to t=2
fitted_meetings = MEETING_DATES[MEETING_DATES<max_t]
model = SimpleJumpsModel(R0, fitted_meetings, INTENSITIES)
curve = simulation.get_curve()

model.fit(curve, shift=0.15)
real_discounts = np.array([curve.discount(tt) for tt in fitted_meetings])
model_discounts = np.array([model.discount(tt) for tt in fitted_meetings])

# plotting
fig = go.Figure()
fig.add_trace(go.Scatter(x=fitted_meetings, y=real_discounts, mode='lines+markers', name='Real Discounts', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=fitted_meetings, y=model_discounts.flatten(), mode='lines+markers', name='Model Discounts', line=dict(color='red', dash='dash')))
fig.update_layout(title='Real vs Model Discounts', xaxis_title='Meeting Dates', yaxis_title='Discount Factor')
fig.show()

model.get_probs()

### We are not ready yet

After all this modeling, the idea would be to fit our model to the observed discount factors or forwards rates in order to infer the probabilities for each meeting. The problem is that without further assumptions, if we belive that there are more than two possible states for the jumps (e.g., up, down, stay), our model is underdetermined: there are infinite combinations of probabilities that can fit the same yield curve, so we need to impose further restrictions. 

A possible approach to overcome this is to impose a particular probability distribution (under $\mathbb{Q}$) for the jumps. A approach that can be found in the literature uses an auxiliary diffusion process that shifts the probabilities. This process can be defined as
$$    dX_t = f(t, X_t) dW_t^{\mathbb Q} $$
where $f(t, X_t)$ is a differentiable and continuous function that controls the volatility of the process. With this, we can define the distribution $\mathbb{Q}$ of the jumps at meeting $\tau_n$ as a function of $X_{\tau_n}$, for example using a softmax transformation:

$$    \mathbb{Q}(J_n = J^{(i)} \mid \mathcal{F}_{\tau_n}) 
    = \frac{\exp\{X_{\tau_n} \cdot w_i\}}{\sum_{j} \exp\{X_{\tau_n} \cdot w_j\}}, $$
where $w_i$ are learnable parameters that control the influence of the diffusion process on the jump probabilities. Using the Radon-Nikodym derivative, we can then relate the real-world probabilities to the risk-neutral ones, allowing us to calibrate the model to market data while capturing the dynamics of policy rate decisions. The likelihodd process defined as
$$
    L_t = \frac{d\mathbb{Q}}{d\mathbb{P}} \Big|_{\mathcal{F}_t} = \exp\left(-\int_0^t \frac{\mu(s, X_s)}{f(s, X_s)} dW_s^{\mathbb P} - \frac{1}{2} \int_0^t \left(\frac{\mu(s, X_s)}{f(s, X_s)}\right)^2 ds\right),
$$
where $\mu(t, X_t)$ is a drift function that can also be parameterized and learned from data.

![](#tbl:probs_fit)

## Appendix 1: A brief review of the math of interest rates

First, lets recall some **definitions** and results commonly used in interest rate modeling. The **discount factor** between time $t$ and $T$ is
$$
P(t,T) = \mathbb{E}^{\mathbb{Q}}\left[\exp\left(-\int_t^T r_s\, ds\right) \Big| \mathcal{F}_t\right],
$$
where $r_t$ is the **instantaneous short rate** and $\mathbb Q$ is the **risk-neutral measure**. The **instantaneous forward rate** with maturity $T$ observed at time $t$ is
$$
f(t,T) = -\frac{\partial}{\partial T} \ln P(t,T).
$$
The **market forward rate** between $T_1$ and $T_2$ is
$$
F(t;T_1,T_2) 
= \frac{1}{T_2 - T_1}\left(\frac{P(t,T_1)}{P(t,T_2)} - 1\right)
= \frac{1}{T_2 - T_1}\left(\exp\left(\int_{T_1}^{T_2} f(t,u)\, du\right) - 1\right).
$$

An OIS swap is an agreement where one party pays a fixed rate $K$ at times $T_1, T_2, \ldots, T_n$ and receives the floating overnight rate at the same times. The value of this contract at time $t$ is
$$
V_{\mathrm{OIS}}(t;K) = \sum_{i=1}^n \delta_i P(t,T_i)\big(F(t;T_{i-1},T_i) - K\big),
$$
where $\delta_i$ is the year fraction between $T_{i-1}$ and $T_i$. The fair fixed rate $K^*$ is the rate that makes the value of the swap zero at inception:
$$
V_{\mathrm{OIS}}(t;K^*) = 0.
$$

Using the par-swap relationship,
$$
K^*\sum_{i=1}^n \delta_i P(t,T_i) = 1 - P(t,T_n),
$$
we can solve for $P(t,T_n)$ recursively:
$$
P(t,T_n) = \frac{1 - K^* \sum_{i=1}^{n-1} \delta_i P(t,T_i)}{1 + K^* \delta_n}.
$$

Repeating this for all maturities yields the discount curve, from which forward rates can be obtained using the definitions above.
